# DPM Engine · Metadata Validation

**Purpose.** Validates the parameterisation before it can be used by the calculation, and issues a verdict per framework: `VALID` or `INVALID`. It works in two modes:

- **Snapshot** (`candidate_json` empty): validates a snapshot of the parameterisation tables and records the verdict. It is the mode of a manual run.
- **Candidate** (`candidate_json` informed): validates one DataLoader upload in the raw tables, before it reaches the parameterisation tables. The verdict is returned to `DE_ENR_NB_LGL_REG_DPM_PARAM_LOAD`, which promotes the upload if it is `VALID` and records the verdict.

- Reads one snapshot (`metadata_as_of`; `latest` by default) of the parameterisation tables (`param_datapoint`, `param_fltr`, `param_ctlg` and `param_src_tbl`), or the rows of the upload in the raw tables (`raw_dpm_param_*`, by `dataload_id`).
- Checks the structure of the files, coordinates, filters, expressions, source tables and period formats, and the specific rules of the Calculated, Rollup and Default types, inspecting the schema of the source tables.
- Two checks read business data and can be disabled: one row per key for Calculated variables (`data_check_enabled`) and the period format of the source data (`period_format_data_check_enabled`).
- Every issue is recorded in `dp_lgl_reg_dpm_iss_log` and, in snapshot mode, the verdict in `dp_lgl_reg_dpm_param_vld`. Every `META.*` issue has `ERROR` severity: a single issue makes the framework `INVALID`.

**When to run:** after each load of the parameterisation. The calculation of a new period reuses the verdict while the parameterisation of the framework does not change.

**Prerequisite:** the engine tables exist (`DE_DP_NB_LGL_REG_CREATE_TABLES`, run at deployment).

**Output:** the notebook ends with the validation summary as its exit value (JSON), with the verdict of each framework.

**Location:** workspace *DP FET Legal and Regulatory Reporting*, folder `01_Processes/DPM`, default Lakehouse `DP_FET_LH_LegalAndRegulatoryReporting`. All engine tables carry the prefix `dp_lgl_reg_dpm_`.



## 1. Parameters


In [ ]:
# Parameterisation snapshot. Empty or "latest" = most recent snapshot.
metadata_as_of = "latest"

# Empty = all frameworks present in the selected snapshot.
frameworks_csv = "FCP"

# Candidate upload (batch DE_DP_NB_LGL_REG_DPM_PARAM_LOAD). Empty = snapshot mode. Informed = JSON with the
# dataload_id of each tab of one DataLoader upload:
# {"datapoints": "...", "filters": "...", "catalog": "...", "sources": "..."}.
# In this mode metadata_as_of and frameworks_csv are not used.
candidate_json = ""

# Raw tables loaded by the DataLoader (one per tab of the parameterisation file).
raw_datapoints_table = "raw_dpm_param_datapoint"
raw_filters_table = "raw_dpm_param_fltr"
raw_catalog_table = "raw_dpm_param_ctlg"
raw_source_config_table = "raw_dpm_param_src_tbl"

# Engine tables.
datapoints_table = "dp_lgl_reg_dpm_param_datapoint"
filters_table = "dp_lgl_reg_dpm_param_fltr"
catalog_table = "dp_lgl_reg_dpm_param_ctlg"
issue_catalog_table = "dp_lgl_reg_dpm_param_iss_ctlg"
issue_log_table = "dp_lgl_reg_dpm_iss_log"
validation_table = "dp_lgl_reg_dpm_param_vld"

# The validation is a strict gate: every META issue has ERROR severity.
# True = the notebook ends with an error when any verdict is INVALID. In candidate mode it
# stays False: DE_ENR_NB_LGL_REG_DPM_PARAM_LOAD needs the verdict and is the one that fails.
fail_notebook_on_invalid_metadata = False

# Maximum source variables per datapoint.
MAX_VARIABLES_PER_DATAPOINT = 70

# Source table configuration (same SCD table as Processing).
source_config_table = "dp_lgl_reg_dpm_param_src_tbl"

# Data check: each Calculated variable without an explicit aggregation returns at most
# one row per grouping key. Period YYYYMM; empty = latest period with data in each table.
data_check_enabled = True
data_check_reference_period = ""

# Data check: all the values of the period column of each source table can be read with
# the Period Format of the Sources tab. Only the distinct values of the column are read.
period_format_data_check_enabled = True


## Common definitions

Helpers shared with the other notebooks of the engine (`DE_ENR_NB_LGL_REG_DPM_COMMON`).


In [ ]:
%run DE_ENR_NB_LGL_REG_DPM_COMMON

## 3. Common helpers and SCD snapshot resolution


In [ ]:
import ast
import hashlib
import json
import re
import uuid
from collections import Counter, defaultdict
from datetime import datetime, timezone
from typing import Any, Iterable

import pandas as pd
from notebookutils import mssparkutils
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark.conf.set("spark.ms.autotune.enabled", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")

ALLOWED_TYPES = {"AGGREGATED", "CALCULATED", "ROLLUP", "DEFAULT"}
ALLOWED_FILTER_OPERATORS = {
    "=", "EQUAL", "EQUALS", "IN", "NOT IN", "NOTIN",
    "LIKE", "NOT LIKE", "NOTLIKE",
    # Range operators.
    ">", ">=", "<", "<=", "BETWEEN",
}
# Periods are checked with canonical_period(), which applies the same syntax as
# resolve_period() in Processing (shared block below).
NUMERIC_TYPES = {
    "tinyint", "smallint", "int", "bigint", "float", "double", "decimal",
}
SUPPORTED_PERIOD_TYPES = {"numeric", "string", "date", "timestamp"}

REQUIRED_COLUMNS = {
    "datapoints": {
        "src_file_nm", "framework_cd", "datapoint_id", "var_cd", "datapoint_cl_typ",
        "src_tbl_id", "metric_fmla_ln", "having_ln", "per_expr_cd",
    },
    "filters": {
        "src_file_nm", "datapoint_id", "var_cd",
        "src_column_nm", "optr_cd", "fltr_val_ln",
    },
    "catalog": {
        "src_file_nm", "datapoint_id", "framework_cd",
        "tmplt_cd", "row_cd", "column_cd",
    },
}

def frame_table(frame):
    """Small DataFrame as a pandas table, with nulls as empty strings."""
    return pd.DataFrame(frame_records(frame), columns=frame.columns)


def read_parameterisation(candidate):
    """Datapoints, filters, catalog and sources to validate, and their snapshot (None for
    a candidate upload, which is not in the parameterisation tables yet)."""
    if candidate:
        return (*candidate_frames(candidate), None)
    metadata_ts = resolve_metadata_as_of(metadata_as_of)
    return (
        scd_snapshot(datapoints_table, metadata_ts),
        scd_snapshot(filters_table, metadata_ts),
        scd_snapshot(catalog_table, metadata_ts),
        scd_snapshot(source_config_table, metadata_ts),
        metadata_ts,
    )


def spark_type_family(data_type):
    """Type family of a column: numeric, string, date or timestamp; other types keep their name."""
    name = data_type.simpleString().split("(")[0].lower()
    if name in NUMERIC_TYPES:
        return "numeric"
    if name in {"string", "char", "varchar"}:
        return "string"
    if name in {"date", "timestamp"}:
        return name
    return name


def stable_issue_key(*parts):
    """Deterministic hash of the given parts."""
    payload = "|".join(clean(x) for x in parts)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def render_template(template, params):
    """Issue message rendered from its template; fails if a parameter is missing."""
    try:
        return template.format(**params)
    except Exception as exc:
        raise RuntimeError(
            f"Issue message template could not be rendered: {template!r}; params={params}"
        ) from exc


# Strict date parsing: a value that does not match the pattern returns NULL.
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")


def canonical_period(label):
    """Normalised form of a period, used to compare periods:
    '', Current, Execution Month, M, M-0 -> 'M'; 'm - 3' -> 'M-3'; 'y-1/12' -> 'Y-1/12'.
    Raises UnsupportedPeriod if the value is not valid."""
    text = re.sub(r"\s+", "", clean(label).upper())
    relative = PERIOD_RELATIVE_PATTERN.fullmatch(text)
    anchored = PERIOD_ANCHORED_PATTERN.fullmatch(text)
    if text in PERIOD_CURRENT_LABELS or (relative and int(relative.group(1)) == 0):
        return "M"
    if relative:
        return f"M-{int(relative.group(1))}"
    if anchored:
        years = int(anchored.group(1) or 0)
        return f"Y-{years}/{anchored.group(2)}" if years else f"Y/{anchored.group(2)}"
    raise UnsupportedPeriod(f"Unsupported period value: {clean(label)!r}")


def safe_canonical_period(label):
    """Canonical form of a period expression, or its cleaned text when it is not supported."""
    try:
        return canonical_period(label)
    except UnsupportedPeriod:
        return clean(label).upper()


## 4. Issue catalog and output tables


In [ ]:
# Columns of the verdict table (dp_lgl_reg_dpm_param_vld).
VALIDATION_SCHEMA = (
    "vld_id STRING, framework_cd STRING, param_snpsh_dt TIMESTAMP, "
    "vld_stts STRING, err_nr BIGINT, crtn_dt TIMESTAMP"
)


def load_issue_catalog():
    """Active codes of the issue catalogue, by code; fails if the table or its columns are missing."""
    if not spark.catalog.tableExists(issue_catalog_table):
        raise RuntimeError(f"Required issue catalog table does not exist: {issue_catalog_table}")
    frame = spark.table(issue_catalog_table)
    if "ctrl_state" in frame.columns:
        frame = frame.where(F.col("ctrl_state") == 1)
    if "row_present_flg" in frame.columns:
        frame = frame.where(F.col("row_present_flg") == 1)
    required = {
        "iss_cd", "dmn_cd", "stg_cd", "pattern_cd", "sev_cd",
        "scp_cd", "msg_tmplt_ln", "param_nm_list_ln", "act_flg",
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise RuntimeError(f"Issue catalog is missing required columns: {missing}")
    rows = [r.asDict(recursive=True) for r in frame.toLocalIterator()]
    catalog = {}
    for row in rows:
        code = clean(row["iss_cd"])
        if not code or code in catalog:
            raise RuntimeError(f"Issue catalog contains blank/duplicate issue_code: {code!r}")
        if clean(row["act_flg"]).upper() in {"0", "FALSE", "F", "N", "NO", ""}:
            continue
        if clean(row["dmn_cd"]) == "META" and clean(row["sev_cd"]).upper() != "ERROR":
            raise RuntimeError(
                f"Metadata issue {code} must use severity ERROR; "
                "VALID_WITH_WARNINGS is intentionally not supported."
            )
        catalog[code] = row
    return catalog


class IssueCollector:
    def __init__(self, catalog, validation_id, metadata_ts):
        """Issue collector of one validation."""
        self.catalog = catalog
        self.validation_id = validation_id
        self.metadata_ts = metadata_ts
        self.rows = []

    def emit(
        self,
        issue_code,
        *,
        framework="",
        source_table="",
        datapoint_id="",
        variable_id="",
        affected_count=0,
        params=None,
        raw_error_class="",
        raw_message="",
    ):
        """Adds one issue, with its message rendered from the catalogue template."""
        if issue_code not in self.catalog:
            raise RuntimeError(f"Unknown issue_code requested by Validator: {issue_code}")
        definition = self.catalog[issue_code]
        params = dict(params or {})
        rendered = render_template(definition["msg_tmplt_ln"], params)
        key = stable_issue_key(
            "VALIDATION", self.validation_id, issue_code, framework, source_table,
            datapoint_id, variable_id, json.dumps(params, sort_keys=True, default=str),
        )
        self.rows.append({
            "iss_id": key,
            "exec_id": "",
            "vld_id": self.validation_id,
            "phs_cd": "VALIDATION",
            "param_snpsh_dt": self.metadata_ts,
            "ref_dt": None,
            "iss_cd": issue_code,
            "sev_cd": clean(definition["sev_cd"]).upper(),
            "framework_cd": framework,
            "src_tbl_id": source_table,
            "datapoint_id": datapoint_id,
            "var_id": variable_id,
            "grp_id": "",
            "afct_nr": int(affected_count or 0),
            "param_json_ln": json.dumps(params, sort_keys=True, separators=(",", ":"), default=str),
            "msg_ln": rendered,
            "raw_err_cla_nm": raw_error_class,
            "raw_msg_ln": raw_message[:4000] if raw_message else "",
            "crtn_dt": utc_now(),
        })

    def persist(self):
        """Writes the collected issues in the issue log; returns the number of rows."""
        if not self.rows:
            return 0
        schema = spark.table(issue_log_table).schema
        values = [
            tuple(row.get(field.name) for field in schema.fields)
            for row in self.rows
        ]
        spark.createDataFrame(values, schema).write.format("delta").mode("append").saveAsTable(issue_log_table)
        return len(self.rows)


## 5. Metadata validation rules


In [ ]:
def metadata_contract(frame, contract_name, collector, frameworks):
    """Checks that a parameterisation table has its required columns; one issue per framework if not."""
    missing = sorted(REQUIRED_COLUMNS[contract_name] - set(frame.columns))
    if not missing:
        return True
    for framework in frameworks or [""]:
        collector.emit(
            "META.CONTRACT.REQUIRED_COLUMN_MISSING",
            framework=framework,
            params={"table_name": contract_name, "columns": ",".join(missing)},
        )
    return False


# Functions not supported inside a metric (META.EXPRESSION.UNSUPPORTED_TOKEN).
UNSUPPORTED_METRIC_FUNCTIONS = {
    "ABS", "ROUND", "COALESCE", "CASE", "WHEN", "THEN", "ELSE", "END",
    "GREATEST", "LEAST", "NVL", "DECODE",
}


def metric_shape(raw):
    """Structure of a metric: aggregation, expression, identifiers and unsupported functions."""
    raw = clean(raw)
    if not raw:
        return {"missing": True}
    wrapper = re.fullmatch(
        r"\s*([A-Za-z_][A-Za-z0-9_]*)\s*\((.*)\)\s*",
        raw,
        flags=re.I | re.S,
    )
    aggregation = None
    body = raw
    if wrapper:
        fn = wrapper.group(1).upper()
        if fn in ALLOWED_AGGREGATIONS:
            aggregation = fn
            body = wrapper.group(2).strip()
        elif fn in UNSUPPORTED_METRIC_FUNCTIONS:
            # GREATEST(A, B) is an unsupported function, not an aggregation.
            return {
                "aggregation": None,
                "body": raw,
                "identifiers": re.findall(r"\b[A-Za-z_][A-Za-z0-9_]*\b", raw),
                "unsupported": [fn],
            }
        else:
            return {"bad_aggregation": fn}
    identifiers = re.findall(r"\b[A-Za-z_][A-Za-z0-9_]*\b", body)
    unsupported_functions = sorted({
        token for token in identifiers
        if token.upper() in UNSUPPORTED_METRIC_FUNCTIONS
    })
    return {
        "aggregation": aggregation,
        "body": body,
        "identifiers": identifiers,
        "unsupported": unsupported_functions,
    }


def validate_arithmetic_syntax(body, identifiers):
    """True if the expression is only arithmetic over the given identifiers."""
    if body == "*":
        return True
    rendered = body
    for token in sorted(set(identifiers), key=len, reverse=True):
        rendered = re.sub(rf"\b{re.escape(token)}\b", "1", rendered, flags=re.I)
    if re.search(r"[^0-9+\-*/().\s]", rendered):
        return False
    try:
        ast.parse(rendered, mode="eval")
        return True
    except Exception:
        return False


def formula_shape(expression):
    """Variables of a formula and whether its syntax is valid."""
    raw = clean(expression).upper()
    variables = tuple(dict.fromkeys(v.upper() for v in LOCAL_VARIABLE_PATTERN.findall(raw)))
    rendered = LOCAL_VARIABLE_PATTERN.sub("1", raw)
    if re.search(r"[^0-9+\-*/().\s]", rendered):
        return variables, False
    try:
        ast.parse(rendered, mode="eval")
        return variables, True
    except Exception:
        return variables, False


def resolve_source(logical_name, source_file=""):
    """The source table is looked up in the Sources tab of the same file."""
    return source_config_for(source_file, logical_name)


SOURCES_REQUIRED_FIELDS = (
    ("phy_tbl_nm", "Physical Table"),
    ("per_column_nm", "Period Column"),
    ("per_fmt_cd", "Period Format"),
)


def sources_row_label(source_file, row):
    """Location of a row of the Sources tab in the messages. The row number is only
    known when the file was loaded with DE_ENR_NB_DPM_Metadata_Load."""
    row_number = clean(row.get("src_row_nr"))
    return f"{source_file}/Sources/row {row_number}" if row_number else f"{source_file}/Sources"


def validate_sources(framework, selected_files, sources_pdf, collector):
    """Rules of the Sources tab of the framework's files.

    - a source table is declared only once per file (META.SOURCE.DUPLICATE);
    - Physical Table, Period Column and Period Format are mandatory
      (META.SOURCE.CONFIG_MISSING);
    - Period Format is DATE or a valid pattern and matches the column type
      (META.SOURCE.PERIOD_FORMAT_INVALID).
    Returns the data checks (period values match the format), which are run at the end
    in run_period_format_data_checks().
    """
    data_checks = {}
    if sources_pdf is None or sources_pdf.empty:
        return data_checks
    rows = sources_pdf[sources_pdf["src_file_nm"].astype(str).isin(selected_files)].to_dict("records")
    counts = Counter(
        (clean(row.get("src_file_nm")), clean(row.get("src_tbl_id")).upper()) for row in rows
    )
    for (source_file, logical), count in sorted(counts.items()):
        if logical and count > 1:
            collector.emit(
                "META.SOURCE.DUPLICATE",
                framework=framework,
                source_table=logical,
                params={"src_tbl_id": logical, "count": count, "src_file_nm": source_file},
            )
    for row in rows:
        source_file = clean(row.get("src_file_nm"))
        logical = clean(row.get("src_tbl_id")).upper()
        if not logical:
            collector.emit(
                "META.SOURCE.CONFIG_MISSING",
                framework=framework,
                params={
                    "src_tbl_id": sources_row_label(source_file, row),
                    "missing_fields": "Source Table",
                },
            )
            continue
        missing = [label for field, label in SOURCES_REQUIRED_FIELDS if not clean(row.get(field))]
        if missing:
            collector.emit(
                "META.SOURCE.CONFIG_MISSING",
                framework=framework,
                source_table=logical,
                params={"src_tbl_id": logical, "missing_fields": ", ".join(missing)},
            )
        physical_table = clean(row.get("phy_tbl_nm"))
        period_format = normalize_period_format(row.get("per_fmt_cd"))
        if not physical_table or not period_format or not spark.catalog.tableExists(physical_table):
            # Non-existent table: reported (META.SOURCE.TABLE_NOT_FOUND) on the variables that use it.
            continue
        schema = {field.name.upper(): field for field in spark.table(physical_table).schema.fields}
        period_field = schema.get(clean(row.get("per_column_nm")).upper())
        if period_field is None:
            # Non-existent column: META.SOURCE.COLUMN_NOT_FOUND on the variables that use it.
            continue
        problem = period_format_problem(period_format, period_field.dataType)
        if problem:
            collector.emit(
                "META.SOURCE.PERIOD_FORMAT_INVALID",
                framework=framework,
                source_table=logical,
                params={"src_tbl_id": logical, "per_fmt_cd": period_format, "detail": problem},
            )
            continue
        if period_format != PERIOD_FORMAT_DATE:
            data_checks.setdefault(
                (physical_table, period_field.name, period_format), []
            ).append((framework, logical))
    return data_checks


def run_period_format_data_checks(data_checks, collector, examples=3):
    """Read the distinct values of the period column of each table and check that all of
    them can be read with the declared format (META.SOURCE.PERIOD_FORMAT_MISMATCH).
    One read per table, column and format."""
    checked = 0
    for (physical_table, period_column, period_format), owners in sorted(data_checks.items()):
        values = (
            spark.table(physical_table)
            .where(F.col(quote_identifier(period_column)).isNotNull())
            .select(
                period_text_expr(period_column).alias("v"),
                period_parsed_expr(period_column, period_format).alias("parsed"),
            )
            .distinct()
        )
        bad = values.where(F.col("parsed").isNull()).select("v").distinct()
        bad_count = bad.count()
        checked += 1
        if not bad_count:
            continue
        sample = [row["v"] for row in bad.orderBy("v").take(examples)]
        for framework, logical in sorted(set(owners)):
            collector.emit(
                "META.SOURCE.PERIOD_FORMAT_MISMATCH",
                framework=framework,
                source_table=logical,
                affected_count=bad_count,
                params={
                    "src_tbl_id": logical,
                    "src_column_nm": period_column,
                    "per_fmt_cd": period_format,
                    "afct_nr": bad_count,
                    "examples": ", ".join(repr(value) for value in sample),
                },
            )
    return checked


VALUE_OPERATORS = {"=", "EQUAL", "EQUALS", "IN", "NOT IN", "NOTIN", ">", ">=", "<", "<=", "BETWEEN"}


def numeric_literal_ok(value):
    """True if the value can be read as a number."""
    try:
        float(value)
        return True
    except ValueError:
        return False


def date_literal_ok(value, family):
    """Dates as yyyy-MM-dd or yyyyMMdd (Processing normalises the latter); timestamp
    columns also accept the time."""
    patterns = ["%Y-%m-%d", "%Y%m%d"]
    if family == "timestamp":
        patterns += ["%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S"]
    for pattern in patterns:
        try:
            datetime.strptime(value, pattern)
            return True
        except ValueError:
            continue
    return False


def validate_rollup(framework, source_file, datapoint_id, rows, grouped, filter_rows, collector):
    """Rules of the Rollup type (the same ones that compile_plan checks in Processing)."""
    defined = []
    if any(clean(row.get("src_tbl_id")) for row in rows):
        defined.append("Source Table")
    if any(clean(row.get("per_expr_cd")) for row in rows):
        defined.append("Period")
    if any(clean(row.get("having_ln")) for row in rows):
        defined.append("Having")
    if any(
        clean(item.get("src_file_nm")) == source_file and clean(item.get("datapoint_id")) == datapoint_id
        for item in filter_rows
    ):
        defined.append("Filters")
    if defined:
        collector.emit(
            "META.ROLLUP.HAS_SOURCE", framework=framework, datapoint_id=datapoint_id,
            params={"datapoint_id": datapoint_id, "fields": ", ".join(defined)},
        )

    expressions = [clean(row.get("metric_fmla_ln")) for row in rows if clean(row.get("metric_fmla_ln"))]
    if len(expressions) != 1:
        collector.emit(
            "META.ROLLUP.EXPRESSION_INVALID", framework=framework, datapoint_id=datapoint_id,
            params={"datapoint_id": datapoint_id, "expression": " | ".join(expressions) or "<empty>"},
        )
        return
    try:
        terms = parse_rollup_expression(expressions[0])
    except ValueError:
        collector.emit(
            "META.ROLLUP.EXPRESSION_INVALID", framework=framework, datapoint_id=datapoint_id,
            params={"datapoint_id": datapoint_id, "expression": expressions[0]},
        )
        return

    children = list(dict.fromkeys(child for child, _ in terms))
    missing = [child for child in children if (source_file, child) not in grouped]
    if missing:
        collector.emit(
            "META.ROLLUP.CHILD_NOT_FOUND", framework=framework, datapoint_id=datapoint_id,
            params={"datapoint_id": datapoint_id, "children": ", ".join(missing)},
        )
        return
    child_types = {
        child: {clean(row.get("datapoint_cl_typ")).upper() for row in grouped[(source_file, child)]}
        for child in children
    }
    defaults = [child for child, types in child_types.items() if "DEFAULT" in types]
    if defaults:
        # A Default is a literal without variables; it cannot be added up in a Rollup.
        collector.emit(
            "META.ROLLUP.CHILD_IS_DEFAULT", framework=framework, datapoint_id=datapoint_id,
            params={"datapoint_id": datapoint_id, "children": ", ".join(defaults)},
        )
        return
    nested = [child for child, types in child_types.items() if "ROLLUP" in types]
    if nested:
        collector.emit(
            "META.ROLLUP.CHILD_IS_ROLLUP", framework=framework, datapoint_id=datapoint_id,
            params={"datapoint_id": datapoint_id, "children": ", ".join(nested)},
        )
        return
    all_types = set().union(*child_types.values())
    if len(all_types) != 1:
        collector.emit(
            "META.ROLLUP.CHILD_TYPE_MIXED", framework=framework, datapoint_id=datapoint_id,
            params={
                "datapoint_id": datapoint_id,
                "children": ", ".join(f"{child} ({'/'.join(sorted(child_types[child]))})" for child in children),
            },
        )
        return

    child_source_rows = [
        row for child in children for row in grouped[(source_file, child)]
        if clean(row.get("src_tbl_id"))
    ]
    periods = {safe_canonical_period(row.get("per_expr_cd")) for row in child_source_rows}
    if len(periods) > 1:
        collector.emit(
            "META.ROLLUP.CHILD_PERIOD_MISMATCH", framework=framework, datapoint_id=datapoint_id,
            params={"datapoint_id": datapoint_id, "periods": ", ".join(sorted(periods))},
        )
    if all_types == {"CALCULATED"}:
        grains = {
            clean((resolve_source(row.get("src_tbl_id"), source_file) or {}).get("grain_id"))
            or clean(row.get("src_tbl_id")).upper()
            for row in child_source_rows
        }
        if len(grains) > 1:
            collector.emit(
                "META.ROLLUP.CHILD_GRAIN_MISMATCH", framework=framework, datapoint_id=datapoint_id,
                params={"datapoint_id": datapoint_id, "grains": ", ".join(sorted(grains))},
            )


def validate_default(framework, source_file, datapoint_id, rows, filter_rows, collector):
    """A Default only has its value in Metric / Formula. Processing publishes it as is in
    each entity and view with results, without reading any table."""
    defined = []
    if any(clean(row.get("src_tbl_id")) for row in rows):
        defined.append("Source Table")
    if any(clean(row.get("var_cd")) for row in rows):
        defined.append("Variable")
    if any(clean(row.get("per_expr_cd")) for row in rows):
        defined.append("Period")
    if any(clean(row.get("having_ln")) for row in rows):
        defined.append("Having")
    if any(
        clean(item.get("src_file_nm")) == source_file and clean(item.get("datapoint_id")) == datapoint_id
        for item in filter_rows
    ):
        defined.append("Filters")
    if defined:
        collector.emit(
            "META.DEFAULT.HAS_SOURCE", framework=framework, datapoint_id=datapoint_id,
            params={"datapoint_id": datapoint_id, "fields": ", ".join(defined)},
        )
    if len(rows) > 1:
        collector.emit(
            "META.DEFAULT.MULTIPLE_ROWS", framework=framework, datapoint_id=datapoint_id,
            params={"datapoint_id": datapoint_id, "count": len(rows)},
        )
    if not any(clean(row.get("metric_fmla_ln")) for row in rows):
        collector.emit(
            "META.DEFAULT.VALUE_MISSING", framework=framework, datapoint_id=datapoint_id,
            params={"datapoint_id": datapoint_id},
        )


def run_cardinality_data_checks(specs, collector):
    """Count, with real data, the rows per key of each Calculated variable without an
    explicit aggregation. If any key has more than one row, the value would be ambiguous
    in Processing: META.CALC.MULTIPLE_ROWS_PER_KEY. One read per table and period, with
    all variables evaluated together (blocks of 250)."""
    by_table = defaultdict(list)
    for spec in specs:
        # Same table with a different period column or format = separate reads.
        by_table[(spec["physical_table"], spec["per_column_nm"], spec.get("per_fmt_cd", ""))].append(spec)
    checked = 0
    for (table_name, _, _), table_specs in sorted(by_table.items()):
        source = spark.table(table_name)
        period_column = table_specs[0]["per_column_nm"]
        period_expr = period_to_date_expr(
            period_column, source.schema[period_column].dataType, table_specs[0].get("per_fmt_cd", "")
        )
        if clean(data_check_reference_period):
            reference_month = validate_reference_period(data_check_reference_period)[:6]
        else:
            latest = source.select(F.max(period_expr).alias("period_date")).first()["period_date"]
            if latest is None:
                continue
            reference_month = f"{latest.year:04d}{latest.month:02d}"
        groups = defaultdict(list)
        for spec in table_specs:
            month = resolve_period(reference_month, spec["period_label"])["physical_month"]
            groups[(month, spec["grain_columns"])].append(spec)
        for (month, grain_columns), group_specs in sorted(groups.items(), key=lambda item: str(item[0])):
            frame = prepare_grain_columns(
                source.where(period_expr == period_literal_to_date(month)), grain_columns
            )
            keys = [grain_internal_column(key) for key, _ in grain_columns]
            for start in range(0, len(group_specs), 250):
                block = group_specs[start:start + 250]
                per_key = frame.groupBy(*keys).agg(*[
                    F.sum(F.when(F.expr(spec["filter_sql"]), F.lit(1)).otherwise(F.lit(0))).alias(f"c{index}")
                    for index, spec in enumerate(block)
                ])
                totals = per_key.agg(*[
                    F.sum(F.when(F.col(f"c{index}") > 1, F.lit(1)).otherwise(F.lit(0))).alias(f"c{index}")
                    for index in range(len(block))
                ]).first()
                for index, spec in enumerate(block):
                    checked += 1
                    affected = int((totals[f"c{index}"] if totals else 0) or 0)
                    if affected:
                        collector.emit(
                            "META.CALC.MULTIPLE_ROWS_PER_KEY",
                            framework=spec["framework_cd"], source_table=table_name,
                            datapoint_id=spec["datapoint_id"], variable_id=spec["var_cd"],
                            params={
                                "var_cd": spec["var_cd"],
                                "datapoint_id": spec["datapoint_id"],
                                "src_tbl_id": table_name,
                                "per_cd": month,
                                "afct_nr": affected,
                            },
                        )
    return checked


def validate_framework(
    framework,
    datapoints_pdf,
    filters_pdf,
    catalog_pdf,
    collector,
    sources_pdf=None,
):
    """Runs all the checks of one framework and records its issues."""
    dp = datapoints_pdf[
        datapoints_pdf["framework_cd"].astype(str).str.upper() == framework
    ].copy()

    if dp.empty:
        collector.emit("META.SELECTION.EMPTY", framework=framework, params={})
        return

    # Catalog and filter rows are scoped by source_file + datapoint because those sheets
    # do not necessarily carry Framework.
    selected_files = set(dp["src_file_nm"].astype(str))
    filters = filters_pdf[filters_pdf["src_file_nm"].astype(str).isin(selected_files)].copy()
    catalog = catalog_pdf[catalog_pdf["src_file_nm"].astype(str).isin(selected_files)].copy()

    filter_rows = filters.to_dict("records")
    catalog_rows = catalog.to_dict("records")

    # Sources tab of the framework's files.
    period_format_checks = validate_sources(framework, selected_files, sources_pdf, collector)
    catalog_by = defaultdict(list)
    for row in catalog_rows:
        catalog_by[(clean(row.get("src_file_nm")), clean(row.get("datapoint_id")))].append(row)

    # Filter relationship checks.
    original_keys = {
        (
            clean(row["src_file_nm"]),
            clean(row["datapoint_id"]),
            clean(row["var_cd"]).upper(),
        )
        for row in dp.to_dict("records")
    }
    seen_filter_signatures = set()
    for row in filter_rows:
        key = (
            clean(row.get("src_file_nm")),
            clean(row.get("datapoint_id")),
            clean(row.get("var_cd")).upper(),
        )
        object_ref = f"{key[0]}/{key[1]}/{key[2] or '<blank>'}"
        if key not in original_keys:
            collector.emit(
                "META.FILTER.ORPHAN",
                framework=framework,
                datapoint_id=key[1],
                variable_id=key[2],
                params={"object_ref": object_ref},
            )
            continue

        column = clean(row.get("src_column_nm"))
        operator = clean(row.get("optr_cd")).upper()
        values = clean(row.get("fltr_val_ln"))
        if not column or not operator or not values:
            collector.emit(
                "META.FILTER.INCOMPLETE",
                framework=framework,
                datapoint_id=key[1],
                variable_id=key[2],
                params={"object_ref": object_ref},
            )
        if operator and operator not in ALLOWED_FILTER_OPERATORS:
            collector.emit(
                "META.FILTER.OPERATOR_INVALID",
                framework=framework,
                datapoint_id=key[1],
                variable_id=key[2],
                params={"operator": operator},
            )

        signature = (key, column.upper(), operator, values)
        if signature in seen_filter_signatures:
            collector.emit(
                "META.FILTER.DUPLICATE",
                framework=framework,
                datapoint_id=key[1],
                variable_id=key[2],
                params={
                    "filter_ref": f"{object_ref}/{column}/{operator}/{values}"
                },
            )
        seen_filter_signatures.add(signature)

    total_source_rows = 0
    mapping_candidates = 0
    data_check_specs = []

    grouped = defaultdict(list)
    for row in dp.to_dict("records"):
        grouped[(clean(row["src_file_nm"]), clean(row["datapoint_id"]))].append(row)

    for (source_file, datapoint_id), rows in sorted(grouped.items()):
        row_types = {clean(row.get("datapoint_cl_typ")).upper() for row in rows}
        invalid_types = sorted(row_types - ALLOWED_TYPES)
        for invalid in invalid_types:
            collector.emit(
                "META.CONTRACT.TYPE_INVALID",
                framework=framework,
                datapoint_id=datapoint_id,
                params={"value": invalid, "datapoint_id": datapoint_id},
            )
        if invalid_types:
            continue

        if len(row_types) != 1:
            collector.emit(
                "META.CONTRACT.MIXED_TYPES",
                framework=framework,
                datapoint_id=datapoint_id,
                params={
                    "datapoint_id": datapoint_id,
                    "types": ",".join(sorted(row_types)),
                },
            )
            continue

        row_type = next(iter(row_types))
        source_rows = [row for row in rows if clean(row.get("src_tbl_id"))]
        formula_rows = [row for row in rows if not clean(row.get("src_tbl_id"))]

        if row_type == "ROLLUP":
            # Specific rules; the Catalog is checked as for the other types.
            validate_rollup(framework, source_file, datapoint_id, rows, grouped, filter_rows, collector)
            source_rows, formula_rows = [], []

        if row_type == "DEFAULT":
            # Specific rules; the Catalog is checked as for the other types.
            validate_default(framework, source_file, datapoint_id, rows, filter_rows, collector)
            source_rows, formula_rows = [], []

        if row_type == "AGGREGATED":
            for row in formula_rows:
                collector.emit(
                    "META.CONTRACT.REQUIRED_VALUE_MISSING",
                    framework=framework,
                    datapoint_id=datapoint_id,
                    variable_id=clean(row.get("var_cd")).upper(),
                    params={
                        "field_name": "Source Table",
                        "object_ref": f"{datapoint_id}/{clean(row.get('var_cd')) or '<blank>'}",
                    },
                )

        source_variables = [clean(row.get("var_cd")).upper() for row in source_rows]
        if len(source_rows) > 1 and any(not value for value in source_variables):
            collector.emit(
                "META.CONTRACT.AMBIGUOUS_VARIABLE",
                framework=framework,
                datapoint_id=datapoint_id,
                params={"datapoint_id": datapoint_id},
            )

        for variable, count in sorted(Counter(v for v in source_variables if v).items()):
            if count > 1:
                collector.emit(
                    "META.CONTRACT.DUPLICATE_VARIABLE",
                    framework=framework,
                    datapoint_id=datapoint_id,
                    variable_id=variable,
                    params={"var_cd": variable, "datapoint_id": datapoint_id},
                )

        if len(source_rows) > MAX_VARIABLES_PER_DATAPOINT:
            collector.emit(
                "META.CONTRACT.TOO_MANY_VARIABLES",
                framework=framework,
                datapoint_id=datapoint_id,
                params={
                    "datapoint_id": datapoint_id,
                    "variable_count": len(source_rows),
                    "max_variables": MAX_VARIABLES_PER_DATAPOINT,
                },
            )

        # Catalog.
        catalog_mapping_valid = False
        catalog_rows_for_dp = catalog_by.get((source_file, datapoint_id), [])
        if not catalog_rows_for_dp:
            collector.emit(
                "META.CATALOG.MISSING",
                framework=framework,
                datapoint_id=datapoint_id,
                params={"datapoint_id": datapoint_id},
            )
        elif len(catalog_rows_for_dp) > 1:
            collector.emit(
                "META.CATALOG.DUPLICATE",
                framework=framework,
                datapoint_id=datapoint_id,
                params={"count": len(catalog_rows_for_dp), "datapoint_id": datapoint_id},
            )
        else:
            cat = catalog_rows_for_dp[0]
            cat_framework = clean(cat.get("framework_cd")).upper()
            if cat_framework != framework:
                collector.emit(
                    "META.CATALOG.FRAMEWORK_MISMATCH",
                    framework=framework,
                    datapoint_id=datapoint_id,
                    params={
                        "catalog_framework": cat_framework,
                        "datapoints_framework": framework,
                        "datapoint_id": datapoint_id,
                    },
                )
            if not all(clean(cat.get(field)) for field in ("tmplt_cd", "row_cd", "column_cd")):
                collector.emit(
                    "META.CATALOG.COORDINATE_INVALID",
                    framework=framework,
                    datapoint_id=datapoint_id,
                    params={
                        "datapoint_id": datapoint_id,
                        "detail": "template, row and column must be non-empty",
                    },
                )
            else:
                catalog_mapping_valid = True
            # Known Value Type (empty = Amount) and TEXT only in Default datapoints.
            raw_value_type = clean(cat.get("val_typ"))
            value_type = re.sub(r"\s+", "", raw_value_type.upper())
            if value_type and value_type not in VALUE_TYPES:
                detail = f"unknown value type {raw_value_type!r}"
            elif value_type == "TEXT" and row_type != "DEFAULT":
                detail = "TEXT is only allowed for Default datapoints"
            else:
                detail = ""
            if detail:
                collector.emit(
                    "META.CATALOG.VALUE_TYPE_INVALID",
                    framework=framework,
                    datapoint_id=datapoint_id,
                    params={"datapoint_id": datapoint_id, "value": raw_value_type, "detail": detail},
                )

        # Formula structure.
        if row_type == "CALCULATED":
            if len(formula_rows) > 1:
                collector.emit(
                    "META.CALC.MULTIPLE_FORMULA_ROWS",
                    framework=framework,
                    datapoint_id=datapoint_id,
                    params={"datapoint_id": datapoint_id},
                )

            if len(source_rows) > 1 and not formula_rows:
                collector.emit(
                    "META.EXPRESSION.MISSING",
                    framework=framework,
                    datapoint_id=datapoint_id,
                    params={
                        "expression_type": "CALCULATED_FORMULA",
                        "object_ref": datapoint_id,
                    },
                )

            if len(formula_rows) == 1:
                formula = clean(formula_rows[0].get("metric_fmla_ln"))
                if not formula:
                    collector.emit(
                        "META.EXPRESSION.MISSING",
                        framework=framework,
                        datapoint_id=datapoint_id,
                        params={
                            "expression_type": "CALCULATED_FORMULA",
                            "object_ref": datapoint_id,
                        },
                    )
                else:
                    referenced, syntax_ok = formula_shape(formula)
                    declared = {value for value in source_variables if value}
                    unknown = sorted(set(referenced) - declared)
                    if unknown:
                        collector.emit(
                            "META.CALC.UNKNOWN_VARIABLE",
                            framework=framework,
                            datapoint_id=datapoint_id,
                            params={
                                "datapoint_id": datapoint_id,
                                "var_cd_list": ",".join(unknown),
                            },
                        )
                    if not syntax_ok:
                        collector.emit(
                            "META.EXPRESSION.UNSUPPORTED_TOKEN",
                            framework=framework,
                            datapoint_id=datapoint_id,
                            params={
                                "object_ref": datapoint_id,
                                "expression": formula,
                            },
                        )
                    for unused in sorted(declared - set(referenced)):
                        collector.emit(
                            "META.CALC.UNUSED_VARIABLE",
                            framework=framework,
                            datapoint_id=datapoint_id,
                            variable_id=unused,
                            params={"var_cd": unused, "datapoint_id": datapoint_id},
                        )

                    compatibility = []
                    for source_row in source_rows:
                        local = clean(source_row.get("var_cd")).upper()
                        if local not in set(referenced):
                            continue
                        config = resolve_source(source_row.get("src_tbl_id"), source_file)
                        if config:
                            compatibility.append(
                                (
                                    clean(config.get("grain_id")),
                                    safe_canonical_period(source_row.get("per_expr_cd")),
                                )
                            )
                    if compatibility and len(set(compatibility)) > 1:
                        collector.emit(
                            "META.CALC.COMPATIBILITY_MISMATCH",
                            framework=framework,
                            datapoint_id=datapoint_id,
                            params={"datapoint_id": datapoint_id},
                        )

        if catalog_mapping_valid and source_rows:
            mapping_candidates += 1

        # Source-backed rows.
        for index, row in enumerate(source_rows, 1):
            total_source_rows += 1
            issues_before_row = len(collector.rows)
            original_variable = clean(row.get("var_cd")).upper()
            effective_variable = (
                original_variable
                if original_variable
                else ("V1" if len(source_rows) == 1 else "")
            )
            object_ref = f"{datapoint_id}/{effective_variable or '<blank>'}"
            logical_source = clean(row.get("src_tbl_id")).upper()
            config = resolve_source(logical_source, source_file)

            if not config:
                collector.emit(
                    "META.SOURCE.CONFIG_MISSING",
                    framework=framework,
                    source_table=logical_source,
                    datapoint_id=datapoint_id,
                    variable_id=effective_variable,
                    params={
                        "src_tbl_id": logical_source,
                        "missing_fields": "source configuration",
                    },
                )
                continue

            physical_table = clean(config.get("phy_tbl_nm"))
            if not spark.catalog.tableExists(physical_table):
                collector.emit(
                    "META.SOURCE.TABLE_NOT_FOUND",
                    framework=framework,
                    source_table=logical_source,
                    datapoint_id=datapoint_id,
                    variable_id=effective_variable,
                    params={
                        "src_tbl_id": logical_source,
                        "candidates": physical_table,
                    },
                )
                continue

            source_frame = spark.table(physical_table)
            source_schema = {field.name: field.dataType for field in source_frame.schema.fields}
            source_columns_upper = {name.upper(): name for name in source_schema}

            # Missing grouping columns are not an error (Processing does not group by them);
            # the period column is mandatory.
            required_source_columns = [clean(config.get("per_column_nm"))]
            missing_group = [
                name for name in required_source_columns
                if name and name.upper() not in source_columns_upper
            ]
            if missing_group:
                collector.emit(
                    "META.SOURCE.COLUMN_NOT_FOUND",
                    framework=framework,
                    source_table=physical_table,
                    datapoint_id=datapoint_id,
                    variable_id=effective_variable,
                    params={
                        "columns": ",".join(missing_group),
                        "reference_type": "GROUP/PERIOD",
                        "src_tbl_id": physical_table,
                    },
                )

            period_column = source_columns_upper.get(
                clean(config.get("per_column_nm")).upper()
            )
            if period_column:
                family = spark_type_family(source_schema[period_column])
                if family not in SUPPORTED_PERIOD_TYPES:
                    collector.emit(
                        "META.SOURCE.PERIOD_TYPE_UNSUPPORTED",
                        framework=framework,
                        source_table=physical_table,
                        datapoint_id=datapoint_id,
                        variable_id=effective_variable,
                        params={
                            "src_column_nm": period_column,
                            "src_tbl_id": physical_table,
                            "data_type": family,
                        },
                    )

            # Period syntax: M, M-n, Y/MM, Y-k/MM, Execution Month, Current (same as Processing).
            try:
                canonical_period(row.get("per_expr_cd"))
                period_ok = True
            except UnsupportedPeriod:
                period_ok = False
            if not period_ok:
                collector.emit(
                    "META.PERIOD.INVALID",
                    framework=framework,
                    source_table=physical_table,
                    datapoint_id=datapoint_id,
                    variable_id=effective_variable,
                    params={
                        "period_value": clean(row.get("per_expr_cd")),
                        "object_ref": object_ref,
                    },
                )

            shape = metric_shape(row.get("metric_fmla_ln"))
            if shape.get("missing"):
                collector.emit(
                    "META.EXPRESSION.MISSING",
                    framework=framework,
                    source_table=physical_table,
                    datapoint_id=datapoint_id,
                    variable_id=effective_variable,
                    params={"expression_type": "METRIC", "object_ref": object_ref},
                )
            elif shape.get("bad_aggregation"):
                collector.emit(
                    "META.EXPRESSION.AGGREGATION_INVALID",
                    framework=framework,
                    source_table=physical_table,
                    datapoint_id=datapoint_id,
                    variable_id=effective_variable,
                    params={
                        "aggregation": shape["bad_aggregation"],
                        "object_ref": object_ref,
                    },
                )
            else:
                if shape.get("unsupported"):
                    collector.emit(
                        "META.EXPRESSION.UNSUPPORTED_TOKEN",
                        framework=framework,
                        source_table=physical_table,
                        datapoint_id=datapoint_id,
                        variable_id=effective_variable,
                        params={
                            "object_ref": object_ref,
                            "expression": clean(row.get("metric_fmla_ln")),
                        },
                    )
                unknown_columns = sorted({
                    token
                    for token in shape.get("identifiers", [])
                    if token.upper() not in source_columns_upper
                })
                if unknown_columns and not shape.get("unsupported"):
                    collector.emit(
                        "META.SOURCE.COLUMN_NOT_FOUND",
                        framework=framework,
                        source_table=physical_table,
                        datapoint_id=datapoint_id,
                        variable_id=effective_variable,
                        params={
                            "columns": ",".join(unknown_columns),
                            "reference_type": "METRIC",
                            "src_tbl_id": physical_table,
                        },
                    )
                # In Aggregated datapoints the metric only allows + and -; ratios are built as
                # Calculated datapoints.
                if row_type == "AGGREGATED" and not shape.get("unsupported") and shape.get("body") != "*":
                    for operator in [op for op in ("*", "/") if op in shape.get("body", "")]:
                        collector.emit(
                            "META.EXPRESSION.OPERATOR_NOT_ALLOWED",
                            framework=framework,
                            source_table=physical_table,
                            datapoint_id=datapoint_id,
                            variable_id=effective_variable,
                            params={
                                "operator": operator,
                                "expression_type": "METRIC",
                                "object_ref": object_ref,
                            },
                        )
                if (
                    not unknown_columns
                    and not shape.get("unsupported")
                    and not (
                        (shape.get("aggregation") or "").upper() == "COUNT"
                        and shape.get("body") == "*"
                    )
                    and not validate_arithmetic_syntax(
                        shape.get("body", ""), shape.get("identifiers", [])
                    )
                ):
                    collector.emit(
                        "META.EXPRESSION.INVALID",
                        framework=framework,
                        source_table=physical_table,
                        datapoint_id=datapoint_id,
                        variable_id=effective_variable,
                        params={
                            "expression_type": "METRIC",
                            "object_ref": object_ref,
                            "src_tbl_id": physical_table,
                            "detail": "invalid arithmetic syntax",
                        },
                    )

            # Filters for this exact original Excel row.
            row_filters = [
                item for item in filter_rows
                if clean(item.get("src_file_nm")) == source_file
                and clean(item.get("datapoint_id")) == datapoint_id
                and clean(item.get("var_cd")).upper() == original_variable
            ]
            for filter_row in row_filters:
                column = clean(filter_row.get("src_column_nm"))
                operator = clean(filter_row.get("optr_cd")).upper()
                if not column:
                    continue
                actual_column = source_columns_upper.get(column.upper())
                if not actual_column:
                    collector.emit(
                        "META.SOURCE.COLUMN_NOT_FOUND",
                        framework=framework,
                        source_table=physical_table,
                        datapoint_id=datapoint_id,
                        variable_id=effective_variable,
                        params={
                            "columns": column,
                            "reference_type": "FILTER",
                            "src_tbl_id": physical_table,
                        },
                    )
                    continue
                family = spark_type_family(source_schema[actual_column])
                raw_values = clean(filter_row.get("fltr_val_ln"))
                filter_ref = f"{object_ref}/{column}/{operator}/{raw_values}"
                # Values read according to the operator (same function as Processing): in IN,
                # NOT IN, LIKE and NOT LIKE a value with a comma goes between double quotes.
                try:
                    all_tokens = filter_value_tokens(raw_values, operator)
                except FilterValueError as exc:
                    collector.emit(
                        "META.FILTER.VALUE_SYNTAX_INVALID",
                        framework=framework,
                        source_table=physical_table,
                        datapoint_id=datapoint_id,
                        variable_id=effective_variable,
                        params={"filter_ref": filter_ref, "detail": str(exc)},
                    )
                    continue
                value_tokens = [
                    token for token in all_tokens
                    if token.upper() not in NULL_TOKENS
                ]
                # Single-value operators (=, >, >=, <, <=) take exactly one value; several
                # values are written with IN.
                if operator in SINGLE_VALUE_OPERATORS and len(all_tokens) > 1:
                    collector.emit(
                        "META.FILTER.VALUE_COUNT_INVALID",
                        framework=framework,
                        source_table=physical_table,
                        datapoint_id=datapoint_id,
                        variable_id=effective_variable,
                        params={
                            "filter_ref": filter_ref,
                            "operator": operator,
                            "expected": 1,
                            "found": len(all_tokens),
                        },
                    )
                # BETWEEN requires exactly two values.
                if operator == "BETWEEN" and len(value_tokens) != 2:
                    collector.emit(
                        "META.FILTER.VALUE_COUNT_INVALID",
                        framework=framework,
                        source_table=physical_table,
                        datapoint_id=datapoint_id,
                        variable_id=effective_variable,
                        params={
                            "filter_ref": filter_ref,
                            "operator": operator,
                            "expected": 2,
                            "found": len(value_tokens),
                        },
                    )
                # Each value must match the column type (number in numeric columns, yyyy-MM-dd or
                # yyyyMMdd in date columns).
                if operator in VALUE_OPERATORS and family in {"numeric", "date", "timestamp"}:
                    for token in value_tokens:
                        if (family == "numeric" and not numeric_literal_ok(token)) or (
                            family in {"date", "timestamp"} and not date_literal_ok(token, family)
                        ):
                            collector.emit(
                                "META.FILTER.VALUE_TYPE_INVALID",
                                framework=framework,
                                source_table=physical_table,
                                datapoint_id=datapoint_id,
                                variable_id=effective_variable,
                                params={
                                    "value": token,
                                    "data_type": family,
                                    "src_column_nm": actual_column,
                                },
                            )
                if operator in {"LIKE", "NOT LIKE", "NOTLIKE"} and family == "numeric":
                    collector.emit(
                        "META.FILTER.VALUE_TYPE_INVALID",
                        framework=framework,
                        source_table=physical_table,
                        datapoint_id=datapoint_id,
                        variable_id=effective_variable,
                        params={
                            "value": clean(filter_row.get("fltr_val_ln")),
                            "data_type": family,
                            "src_column_nm": actual_column,
                        },
                    )

            # Calculated variable without explicit aggregation (FIRST) and without metadata errors
            # -> the data check verifies that there are no several rows per key.
            aggregation = (shape.get("aggregation") or "FIRST").upper() if not shape.get("missing") else ""
            if (
                row_type == "CALCULATED"
                and aggregation == "FIRST"
                and len(collector.rows) == issues_before_row
                and period_column
            ):
                data_check_specs.append({
                    "framework_cd": framework,
                    "datapoint_id": datapoint_id,
                    "var_cd": effective_variable,
                    "physical_table": physical_table,
                    "per_column_nm": period_column,
                    "per_fmt_cd": clean(config.get("per_fmt_cd")),
                    "period_label": clean(row.get("per_expr_cd")),
                    "grain_columns": tuple(config.get("grain_columns") or ()),
                    "filter_sql": " AND ".join(
                        f"({build_filter_sql(item, source_schema)})" for item in row_filters
                        if clean(item.get("src_column_nm"))
                    ) or "TRUE",
                })

    if total_source_rows == 0:
        collector.emit("META.OUTPUT.NO_SOURCE_VARIABLES", framework=framework, params={})
    if mapping_candidates == 0:
        collector.emit("META.OUTPUT.NO_MAPPING", framework=framework, params={})
    return data_check_specs, period_format_checks


## 6. Execute validation


In [ ]:
def run_validation():
    """Validates the parameterisation of the requested frameworks; returns the summary.

    Snapshot mode: validates a snapshot of the parameterisation tables and records the
    verdict of each framework in the validation table. Candidate mode: validates one
    DataLoader upload in the raw tables and only returns the verdict. The issues are
    recorded in the issue log in both modes. Fails at the end if a framework is INVALID
    and fail_notebook_on_invalid_metadata is set.
    """
    # Assigned here and read by source_config_for() (helpers cell).
    global SOURCE_CONFIG
    issue_catalog = load_issue_catalog()
    validation_id = str(uuid.uuid4())
    candidate = json.loads(clean(candidate_json) or "{}")
    datapoints, filters, catalog, sources, resolved_metadata_as_of = read_parameterisation(candidate)
    # Source configuration of the same rows as the parameterisation.
    SOURCE_CONFIG = load_source_config(sources)
    scope = "candidate upload" if candidate else f"metadata snapshot {resolved_metadata_as_of}"

    # A candidate upload is validated whole: frameworks_csv only applies to a snapshot.
    requested_frameworks = set() if candidate else set(split_csv(frameworks_csv))
    available_frameworks = {
        clean(row["framework_cd"]).upper()
        for row in datapoints.select("framework_cd").distinct().toLocalIterator()
        if clean(row["framework_cd"])
    }
    frameworks = sorted(requested_frameworks or available_frameworks)
    missing_requested = sorted(requested_frameworks - available_frameworks)
    if missing_requested:
        raise ValueError(
            f"Requested framework(s) are not present in {scope}: {missing_requested}"
        )
    if not frameworks:
        raise ValueError(f"No frameworks exist in {scope}.")

    collector = IssueCollector(issue_catalog, validation_id, resolved_metadata_as_of)

    contracts_ok = True
    contracts_ok &= metadata_contract(datapoints, "datapoints", collector, frameworks)
    contracts_ok &= metadata_contract(filters, "filters", collector, frameworks)
    contracts_ok &= metadata_contract(catalog, "catalog", collector, frameworks)

    if contracts_ok:
        datapoints_pdf = frame_table(datapoints)
        filters_pdf = frame_table(filters)
        catalog_pdf = frame_table(catalog)
        sources_pdf = frame_table(sources)
        if "src_file_nm" not in sources_pdf.columns:
            sources_pdf["src_file_nm"] = ""

        # Each parameterisation file must contain a single framework.
        for source_file, file_rows in datapoints_pdf.groupby(datapoints_pdf["src_file_nm"].astype(str)):
            file_frameworks = sorted({
                clean(value).upper() for value in file_rows["framework_cd"] if clean(value)
            })
            if len(file_frameworks) > 1:
                for framework in file_frameworks:
                    if framework in frameworks:
                        collector.emit(
                            "META.CONTRACT.MIXED_FRAMEWORK",
                            framework=framework,
                            params={
                                "src_file_nm": source_file,
                                "framework_list": ",".join(file_frameworks),
                            },
                        )

        data_check_specs = []
        period_format_checks = {}
        for framework in frameworks:
            specs, format_checks = validate_framework(
                framework,
                datapoints_pdf,
                filters_pdf,
                catalog_pdf,
                collector,
                sources_pdf,
            ) or ([], {})
            data_check_specs.extend(specs)
            for key, owners in format_checks.items():
                period_format_checks.setdefault(key, []).extend(owners)

        # The values of the period column of each table match the format declared in Sources.
        # Only the distinct values of the column are read.
        period_format_checked = 0
        if period_format_data_check_enabled and period_format_checks:
            period_format_checked = run_period_format_data_checks(period_format_checks, collector)

        # Data check, only for variables without metadata errors (filtered when building
        # data_check_specs).
        data_checked_variables = 0
        if data_check_enabled and data_check_specs:
            data_checked_variables = run_cardinality_data_checks(data_check_specs, collector)

    collector.persist()

    validation_rows = []
    for framework in frameworks:
        error_count = sum(
            1 for row in collector.rows
            if row["framework_cd"] in {"", framework}
            and row["sev_cd"] == "ERROR"
        )
        validation_rows.append({
            "vld_id": validation_id,
            "framework_cd": framework,
            "param_snpsh_dt": resolved_metadata_as_of,
            "vld_stts": "INVALID" if error_count else "VALID",
            "err_nr": int(error_count),
            "crtn_dt": utc_now(),
        })

    verdicts = spark.createDataFrame(validation_rows, VALIDATION_SCHEMA)
    # The verdict of a candidate upload is recorded by DE_ENR_NB_LGL_REG_DPM_PARAM_LOAD, with the
    # snapshot it creates when it promotes the upload.
    if not candidate:
        verdicts.write.format("delta").mode("append").saveAsTable(validation_table)
    display(verdicts.orderBy("framework_cd"))

    issue_summary = (
        spark.table(issue_log_table)
        .where(F.col("vld_id") == validation_id)
        .groupBy("framework_cd", "iss_cd")
        .count()
        .orderBy("framework_cd", "iss_cd")
    )
    display(issue_summary)

    invalid_frameworks = [
        row["framework_cd"] for row in validation_rows if row["vld_stts"] == "INVALID"
    ]
    summary = {
        "vld_id": validation_id,
        "candidate": candidate,
        "param_snpsh_dt": str(resolved_metadata_as_of or ""),
        "frameworks": frameworks,
        "verdicts": {row["framework_cd"]: row["vld_stts"] for row in validation_rows},
        "invalid_frameworks": invalid_frameworks,
        "issue_count": len(collector.rows),
    }
    print(json.dumps(summary, indent=2))

    if invalid_frameworks and bool(fail_notebook_on_invalid_metadata):
        raise RuntimeError(
            f"Metadata validation failed for framework(s): {invalid_frameworks}"
        )
    return summary


validation_result = run_validation()
# Exit value for the notebook or batch that launches this one (validation summary, as JSON).
mssparkutils.notebook.exit(json.dumps(validation_result))
